# LeetCode 4028 — Minimum Operations to Make a Rotated Palindrome II bằng Python

- Link: https://leetcode.com/problems/minimum-operations-to-make-a-rotated-palindrome-ii/
- Độ khó: **Hard**
- Chủ đề: Toán, Chuỗi, Tích chập (convolution), FFT, Biến đổi Fourier rời rạc (DFT) trên vòng tròn 26 chữ cái

> **Bài Premium:** đề bài viết theo hiểu biết về bài, nên đối chiếu lại nếu có tài khoản Premium (đề thực tế lấy từ bản doocs).

Skeleton (Python3):

```python
class Solution:
    def minOperations(self, s: str) -> int:
```

## 1. Giới thiệu & mục tiêu học

Cho chuỗi `s` gồm chữ thường (`2 ≤ |s| ≤ 5·10^4`). Có hai phép: **tăng** một ký tự lên chữ cái kế tiếp (sau `z` là `a`) và **xoay trái** (đưa ký tự đầu xuống cuối). Tìm số phép **ít nhất** để chuỗi thành palindrome.

Với `n` nhỏ ta thử mọi số lần xoay `k` rồi ghép cặp ký tự đối xứng. Nhưng `n = 5·10^4` làm `O(n²)` (cỡ `10^9` phép) quá chậm. Điểm mấu chốt: tổng chi phí ghép cặp **chỉ phụ thuộc tổng chỉ số của mỗi cặp**, và việc tính chi phí cho **mọi tổng chỉ số** cùng lúc chính là một **tích chập**, giải được bằng FFT.

Sau notebook này bạn sẽ:

1. Chuyển bài toán thành: `min_k ( k + chi_phí_ghép_cặp_sau_k_lần_xoay )`.
2. Hiểu chi phí biến hai chữ cái thành bằng nhau là **khoảng cách cung ngắn** `min(d, 26 - d)` trên vòng tròn chữ cái.
3. Nhận ra các cặp đối xứng sau `k` lần xoay đều có **tổng chỉ số gốc** `≡ 2k + n - 1 (mod n)` và tính chi phí cho mọi tổng chỉ số bằng tích chập.
4. Dùng FFT (numpy) và mẹo **chéo hoá hạt nhân chi phí bằng DFT** (chỉ cần 14 tần số), và một cách **không dùng numpy** bằng nhân số nguyên lớn.
5. Kiểm chứng mọi cách bằng BFS trên chuỗi thật (bảng chữ cái thu nhỏ) và test ngẫu nhiên có seed.

## 2. Đề bài & ví dụ

**Phát biểu lại.** Bạn có thể dùng, theo thứ tự tuỳ ý và số lần tuỳ ý:

- **Tăng:** chọn chỉ số `i`, thay `s[i]` bằng chữ cái kế tiếp (`'z'` → `'a'`).
- **Xoay trái:** chuyển ký tự đầu xuống cuối chuỗi (`"abc" → "bca"`).

Trả về số phép **ít nhất** để `s` thành **palindrome** (đọc xuôi ngược như nhau).

**Ví dụ 1:** `s = "abc"` → `2`. Xoay trái: `"bca"`; tăng `'a'` thành `'b'`: `"bcb"` là palindrome. Tổng 2.

**Ví dụ 2:** `s = "yb"` → `3`. Tăng `'y'` ba lần: `y → z → a → b` thành `"bb"`.

**Ràng buộc**

- `2 <= s.length <= 5 * 10^4`
- `s` chỉ gồm chữ thường.

In [1]:
EXAMPLES = [("abc", 2), ("yb", 3)]
for s, e in EXAMPLES:
    print(f"s = {s!r} -> kỳ vọng {e}")

s = 'abc' -> kỳ vọng 2
s = 'yb' -> kỳ vọng 3


## 3. Kiến thức nền

### 3.1. Chữ cái thành số: `ord(c) - ord('a')`

Ta đánh số `a = 0, b = 1, ..., z = 25`. Phép "tăng" khi đó là `x → (x + 1) % 26`.

In [2]:
s = "yb"
nums = [ord(c) - ord("a") for c in s]
print(nums)                       # [24, 1]
print([(x + 1) % 26 for x in nums])   # tăng cả hai: y->z (25), b->c (2)
print((25 + 1) % 26)              # z -> a

[24, 1]
[25, 2]
0


### 3.2. Khoảng cách trên vòng tròn 26 chữ cái

Để hai chữ cái `x`, `y` bằng nhau, ta chỉ cần **tăng** một trong hai (không cần giảm). Muốn `x → y` phải tăng `(y - x) % 26` lần; muốn `y → x` phải tăng `(x - y) % 26` lần. Hai số này cộng lại bằng 26 (nếu `x ≠ y`), và ta chọn số nhỏ hơn:

`cost(x, y) = min(d, 26 - d)` với `d = |x - y|`.

Giống kim đồng hồ: chọn đường ngắn hơn trên vòng tròn.

In [3]:
def cyc_cost(x, y, A=26):
    d = abs(x - y)
    return min(d, A - d)

print(cyc_cost(24, 1))   # y và b: d=23 -> min(23, 3) = 3
print(cyc_cost(0, 25))   # a và z: 1
print(cyc_cost(0, 13))   # a và n: xa nhất, 13
print([cyc_cost(0, z) for z in range(26)])

3
1
13
[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 12, 11, 10, 9, 8, 7, 6, 5, 4, 3, 2, 1]


### 3.3. Palindrome và các cặp đối xứng

Chuỗi `t` độ dài `n` là palindrome ⇔ `t[i] == t[n-1-i]` với mọi `i < n // 2`. Ký tự giữa (khi `n` lẻ) tự do. Vậy số cặp cần ghép là `n // 2`.

In [4]:
t = "abcba"
n = len(t)
print([(i, n - 1 - i, t[i] == t[n - 1 - i]) for i in range(n // 2)])
t = "abcab"
print([(i, n - 1 - i, t[i] == t[n - 1 - i]) for i in range(n // 2)])

[(0, 4, True), (1, 3, True)]
[(0, 4, False), (1, 3, False)]


### 3.4. Xoay trái `k` lần và công thức chỉ số

Sau `k` lần xoay trái, ký tự ở vị trí `i` của chuỗi mới chính là ký tự ở vị trí `(i + k) % n` của chuỗi gốc. Lấy lát cắt: `s[k:] + s[:k]`.

In [5]:
s = "abcde"
n = len(s)
for k in range(n):
    rotated = s[k:] + s[:k]
    check = "".join(s[(i + k) % n] for i in range(n))
    print(k, rotated, check, rotated == check)

0 abcde abcde True
1 bcdea bcdea True
2 cdeab cdeab True
3 deabc deabc True
4 eabcd eabcd True


### 3.5. Tích chập (convolution) theo "tổng chỉ số"

Với hai dãy `a`, `b`, tích chập là dãy `conv[m] = Σ_{i + j = m} a[i] · b[j]`. Với **tích chập vòng** độ dài `n`, ta lấy `i + j ≡ m (mod n)` (gấp phần `m ≥ n` về lại `m - n`).

Ý nghĩa với bài này: đặt `a[i] = 1` nếu `s[i] = 'x'`, `b[j] = 1` nếu `s[j] = 'y'`. Khi đó `conv[m]` **đếm số cặp** `(i, j)` có `i + j = m` mà `s[i] = x`, `s[j] = y`.

In [6]:
def conv_naive(a, b):
    out = [0] * (len(a) + len(b) - 1)
    for i, x in enumerate(a):
        for j, y in enumerate(b):
            out[i + j] += x * y
    return out

s = "abcab"
ia = [1 if ch == "a" else 0 for ch in s]
ib = [1 if ch == "b" else 0 for ch in s]
print(ia, ib)
c = conv_naive(ia, ib)
print(c)                       # đếm cặp (i, j): s[i]='a', s[j]='b', i + j = m
n = len(s)
circ = [c[m] + (c[m + n] if m + n < len(c) else 0) for m in range(n)]
print(circ)                    # gấp mod n

[1, 0, 0, 1, 0] [0, 1, 0, 0, 1]
[0, 1, 0, 0, 2, 0, 0, 1, 0]
[0, 1, 1, 0, 2]


### 3.6. FFT làm tích chập nhanh: `numpy.fft`

Định lý: DFT của tích chập là tích **từng phần tử** của DFT. Vậy `conv = ifft(fft(a) * fft(b))`, tốn `O(n log n)` thay vì `O(n²)`. Hai lưu ý:

1. **Đệm số 0** (zero-padding) tới độ dài `≥ len(a) + len(b) - 1` để không bị "quấn vòng" ngoài ý muốn.
2. Kết quả là số thực có sai số rất nhỏ; ta `round` về số nguyên.

In [7]:
import numpy as np

a = [1, 2, 3]
b = [0, 1, 0.5]
size = 8                                # >= 3 + 3 - 1, luỹ thừa của 2 cho nhanh
fa = np.fft.fft(a, size)                # tự đệm số 0
fb = np.fft.fft(b, size)
conv_fft = np.fft.ifft(fa * fb).real
print(np.round(conv_fft[:5], 6))
print(conv_naive(a, b))

[0.  1.  2.5 4.  1.5]
[0, 1, 2.5, 4.0, 1.5]


### 3.7. Số phức `e^{iθ}` và `cos` (chỉ cần nhớ hai điều)

`e^{iθ} = cos θ + i sin θ`, và `e^{iα} · conj(e^{iβ}) = e^{i(α - β)}`. Nhờ đó `Σ e^{iθ(x_i − x_j)}` biến **hiệu** hai chữ cái thành một **pha**. Ta sẽ dùng điều này ở Cách 4 để biến hàm chi phí `min(d, 26 − d)` thành một tổng các `cos`.

In [8]:
import cmath, math
theta = 2 * math.pi * 3 / 26
x, y = 5, 2
lhs = cmath.exp(1j * theta * x) * cmath.exp(1j * theta * y).conjugate()
rhs = cmath.exp(1j * theta * (x - y))
print(abs(lhs - rhs) < 1e-12, round(lhs.real, 6), round(math.cos(theta * (x - y)), 6))

True -0.568065 -0.568065


## 4. Ý tưởng nền tảng / trực giác

**Tính tay ví dụ 1: `s = "abc"`**, `n = 3`. Ta thử `k = 0, 1, 2` lần xoay trái:

| k | chuỗi sau xoay | cặp (đầu, cuối) | chi phí ghép cặp | tổng `k + chi phí` |
|---|---|---|---|---|
| 0 | `abc` | (a, c) | `min(2, 24) = 2` | 2 |
| 1 | `bca` | (b, a) | `1` | `1 + 1 = 2` |
| 2 | `cab` | (c, b) | `1` | `2 + 1 = 3` |

Nhỏ nhất là **2**. Nhận xét: xoay nhiều hơn `n - 1` lần là vô ích (xoay `n` lần về lại chuỗi cũ).

**Có cần xét thứ tự các phép?** Tăng một ký tự là phép **theo từng ký tự gốc**, xoay chỉ thay đổi vị trí. Dù xen kẽ thế nào, kết quả cuối là: chuỗi gốc xoay `k` lần (`k` = tổng số phép xoay) với một số ký tự đã được tăng. Vì vậy tổng chi phí tối thiểu = `k + tổng chi phí ghép cặp` (ta kiểm chứng lại bằng BFS ở phần 5).

## 5. Cách 1 — Vét cạn: BFS trên chuỗi và công thức theo từng `k`

**Cách 1a — BFS nguyên văn.** Trạng thái là chuỗi; mỗi bước áp dụng "xoay trái" hoặc "tăng ký tự `i`". BFS cho số phép ít nhất tới một palindrome. Số trạng thái `26^n`, nên ta **khái quát hoá bảng chữ cái** thành `A` chữ (mặc định 26) và test với `A` nhỏ (3–6) để BFS kịp chạy với `n ≤ 6`.

**Cách 1b — Công thức theo `k`.** Với mỗi `k = 0..n-1`: chi phí `k + Σ_{i < n/2} cost(s[(i+k)%n], s[(n-1-i+k)%n])`. Tốn `O(n)` mỗi `k`, tổng `O(n²)`.

In [9]:
from collections import deque

def min_ops_bfs(s, A=26, right_rotate=False):
    n = len(s)
    start = tuple(ord(c) - 97 for c in s)
    def is_pal(t):
        return all(t[i] == t[n - 1 - i] for i in range(n // 2))
    dist = {start: 0}
    queue = deque([start])
    while queue:
        t = queue.popleft()
        if is_pal(t):
            return dist[t]
        nxt = [t[1:] + t[:1]]                       # xoay trái
        if right_rotate:
            nxt.append(t[-1:] + t[:-1])             # (chỉ dùng cho biến thể)
        for i in range(n):                          # tăng ký tự i
            nxt.append(t[:i] + ((t[i] + 1) % A,) + t[i + 1:])
        for u in nxt:
            if u not in dist:
                dist[u] = dist[t] + 1
                queue.append(u)

def to_nums(s):
    return [ord(c) - 97 for c in s]

def min_ops_formula(s, A=26):
    n = len(s)
    a = to_nums(s)
    best = float("inf")
    for k in range(n):
        total = k
        for i in range(n // 2):
            total += cyc_cost(a[(i + k) % n], a[(n - 1 - i + k) % n], A)
        best = min(best, total)
    return best

for s, e in EXAMPLES:
    print(s, min_ops_bfs(s), min_ops_formula(s), e)

abc 2 2 2
yb 3 3 3


**Độ phức tạp.** BFS: `O(A^n · n)`. Công thức: `O(n²)`; với `n = 5·10^4` là `~ 6·10^8` phép `cyc_cost` — quá chậm trong Python.

**Nút thắt:** với mỗi `k` ta ghép lại `n/2` cặp từ đầu, trong khi các cặp (theo chỉ số gốc) lặp lại giữa các `k` khác nhau.

## 6. Bước đột phá

### 6.1. Các cặp đối xứng được xác định bởi một **tổng chỉ số**

Sau `k` lần xoay, cặp `(i, n-1-i)` ứng với chỉ số gốc `a = (i + k) % n` và `b = (n - 1 - i + k) % n`. Tổng:

`a + b ≡ 2k + n - 1 (mod n)`,

**không phụ thuộc `i`**. Đặt `c = (2k + n - 1) % n`. Ngược lại, mọi cặp chỉ số gốc phân biệt `{a, b}` với `a + b ≡ c (mod n)` đều là một cặp đối xứng. Số cặp là `n // 2` — khớp.

### 6.2. Chi phí cho **mọi** `c` tính một lần

Đặt `P[c] = Σ_{ {a,b}: a < b, (a+b) mod n = c } cyc_cost(s[a], s[b])`. Khi đó

`đáp án = min_{k=0..n-1} ( k + P[(2k + n - 1) % n] )`.

### 6.3. `P` là một tích chập

Cộng theo cặp chữ cái `(x, y)`: `P[c] = ½ Σ_{x,y} cost(x, y) · conv_vòng(I_x, I_y)[c]`, trong đó `I_x[i] = 1` nếu `s[i] = x`. Hệ số `½` vì cặp có thứ tự `(a, b)` và `(b, a)` đều được đếm; cặp `a = b` có chi phí 0 nên không ảnh hưởng.

Có `26 × 26` cặp chữ cái nhưng chỉ `26` biến đổi Fourier (mỗi `I_x` một lần) cộng thêm phép nhân từng phần tử.

### 6.4. Chéo hoá hạt nhân chi phí bằng DFT (mẹo của lời giải tham khảo)

Hàm `cost(d)` là hàm **chẵn** và **tuần hoàn** trên `Z/26Z`. Khai triển Fourier: `cost(d) = (1/26) Σ_t ĉ(t) · cos(2π t d / 26)`, với `ĉ(t) = Σ_z cost(z) cos(2π t z / 26)`. Khi đó `Σ_{(a,b)} cost(s[a] − s[b])` trở thành `Σ_t ĉ(t)/26 · Σ_{(a,b)} Re e^{iθ_t (s[a] − s[b])}`, và tổng trong cùng là **một tích chập** của dãy phức `A_t[i] = e^{iθ_t s[i]}` với `conj(A_t)`. Do tính chẵn, chỉ cần `t = 0..13` (các `t` còn lại là đối xứng), ta còn **14 tích chập** thay vì 26.

**Quy tắc nếu… thì…**

- Nếu cần tổng theo tổng-chỉ-số ⇒ nghĩ tới tích chập.
- Nếu hạt nhân chi phí chỉ phụ thuộc hiệu `x − y` trên vòng tròn ⇒ DFT chéo hoá nó.
- Bước cuối vẫn là `min_k (k + P[...])`.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Gom theo tổng chỉ số, `O(n²)` nhưng chỉ một lần

Duyệt mọi cặp `i < j`, cộng chi phí vào `P[(i + j) % n]`. Sau đó đáp án = `min_k (k + P[(2k+n-1) % n])`. Cùng độ phức tạp `O(n²)` nhưng cấu trúc đã sẵn sàng để thay vòng lặp cặp bằng tích chập.

In [10]:
def min_ops_group(s, A=26):
    n = len(s)
    a = to_nums(s)
    P = [0] * n
    for i in range(n):
        for j in range(i + 1, n):
            P[(i + j) % n] += cyc_cost(a[i], a[j], A)
    return min(k + P[(2 * k + n - 1) % n] for k in range(n))

for s, e in EXAMPLES:
    print(s, min_ops_group(s), e)

abc 2 2
yb 3 3


### 7.2. Cách 3 — FFT với 26 mảng chỉ báo (ma trận chi phí tường minh)

**Giải thích từng đoạn.**

1. Với mỗi chữ cái `x`, mảng `I_x` (độ dài `size ≥ 2n`) có `1` ở các vị trí `s[i] = x`.
2. `F_x = rfft(I_x)` (FFT cho dãy thực).
3. Tổng cần tính trong miền tần số: `T = Σ_x F_x · ( Σ_y cost(x,y) · F_y )`. Mỗi `x` tốn `26` phép cộng nhân mảng, tổng `26 × 26 × (size/2)` phép phức.
4. `conv = irfft(T)`; gấp `conv[m] + conv[m + n]` để được tích chập vòng; làm tròn; chia 2.

In [11]:
def min_ops_fft_matrix(s, A=26):
    n = len(s)
    size = 1
    while size < 2 * n:
        size <<= 1
    nums = np.array(to_nums(s))
    F = []
    for x in range(A):
        ind = (nums == x).astype(float)
        F.append(np.fft.rfft(ind, size))
    total = np.zeros(size // 2 + 1, dtype=complex)
    for x in range(A):
        G = sum(cyc_cost(x, y, A) * F[y] for y in range(A))
        total += F[x] * G
    conv = np.fft.irfft(total, size)
    ordered = np.rint(conv[:n] + conv[n:2 * n]).astype(np.int64)    # cặp có thứ tự
    P = ordered // 2
    return min(k + int(P[(2 * k + n - 1) % n]) for k in range(n))

for s, e in EXAMPLES:
    print(s, min_ops_fft_matrix(s), e)

abc 2 2
yb 3 3


**Độ phức tạp.** `O(A² · n log n)` — với `A = 26` là `~676 · n log n`; `n = 5·10^4` chạy ~0.2 giây với numpy.

### 7.3. Cách 4 — FFT với 14 tần số (DFT chéo hoá hạt nhân) — bản nộp

**Giải thích từng đoạn.**

1. `ĉ(t) = Σ_z cost(z) · cos(2π t z / A)` cho `t = 0..A`.
2. Với mỗi `t = 0..A//2`: `θ = 2π t / A`; dãy phức `a[i] = e^{iθ · s[i]}`, `b = conj(a)`. Tích chập `a * b` có `conv[m] = Σ_{i + j = m} e^{iθ (s[i] − s[j])}`.
3. Gộp `conv[m] + conv[m + n]` (tổng mod `n`), nhân `ĉ(t)` và hệ số `mult` (= 1 nếu `t = 0` hoặc `t = A/2`, ngược lại 2 vì `t` và `A − t` trùng nhau).
4. Cộng dồn vào `dp[c]`. Khi xong, `dp[c] / (2A)` là `P[c]` (chia `A` của DFT và 2 của cặp có thứ tự: với `A = 26` là `/52`).
5. `đáp án = min_k (k + round(dp[(2k+n-1) % n] / (2A)))`.

In [12]:
def min_ops_fft_dft(s, A=26):
    n = len(s)
    size = 1
    while size < 2 * n:
        size <<= 1
    nums = np.array(to_nums(s), dtype=np.int64)
    chat = np.zeros(A + 1)
    for t in range(A + 1):
        for z in range(A):
            chat[t] += min(z, A - z) * math.cos(2 * math.pi * t * z / A)
    dp = np.zeros(n)
    for t in range(A // 2 + 1):
        theta = 2 * math.pi * t / A
        a = np.exp(1j * theta * nums)
        fa = np.fft.fft(a, size)
        fb = np.fft.fft(np.conj(a), size)
        conv = np.fft.ifft(fa * fb).real
        mult = 1 if (t == 0 or (A % 2 == 0 and t == A // 2)) else 2
        dp += mult * chat[t] * (conv[:n] + conv[n:2 * n])
    best = float("inf")
    for k in range(n):
        best = min(best, k + round(dp[(2 * k + n - 1) % n] / (2 * A)))
    return best

for s, e in EXAMPLES:
    print(s, min_ops_fft_dft(s), e)

abc 2 2
yb 3 3


**Độ phức tạp.** `O(A · n log n)` cho các FFT (14 tần số khi `A = 26`), bộ nhớ `O(n)`. `chat` tính một lần `O(A²)`.

**Về sai số.** Kết quả thực có sai số cỡ `10^-9` so với số nguyên cần tìm (kiểm tra ở mục 11); phép `round` an toàn vì ta chia `52` một giá trị tổng nguyên `≤ 26 · n · 13`.

### 7.4. Cách 5 — Tích chập **chính xác**, không dùng numpy: nhân số nguyên lớn (Kronecker)

Python nhân số nguyên lớn khá nhanh (Karatsuba). Mẹo "Kronecker": gói dãy `a[0..n-1]` thành số nguyên `Σ a[i] · 2^{24·i}`. Tích hai số như vậy chính là gói của **tích chập** `conv`, miễn là mỗi hệ số `< 2^24`. Ở đây hệ số tối đa `≤ 13 · n ≈ 6.5·10^5 < 2^24 ≈ 1.6·10^7` nên đủ chỗ.

Thuật toán: `T = Σ_x I_x · G_x` với `I_x` gói chỉ báo, `G_x = Σ_y cost(x, y) · I_y` (cộng có hệ số số nguyên nhỏ). Sau đó tách 3 byte một từ kết quả. Hoàn toàn **số nguyên**, không sai số làm tròn, chạy ~2–3 giây cho `n = 5·10^4`.

In [13]:
def min_ops_bigint(s, A=26):
    n = len(s)
    W = 3                                       # 3 byte = 24 bit mỗi hệ số
    a = to_nums(s)
    ind = []
    for x in range(A):
        buf = bytearray(W * n)
        for i, v in enumerate(a):
            if v == x:
                buf[W * i] = 1
        ind.append(int.from_bytes(buf, "little"))
    total = 0
    for x in range(A):
        G = 0
        for y in range(A):
            c = cyc_cost(x, y, A)
            if c:
                G += c * ind[y]
        total += ind[x] * G
    raw = total.to_bytes(W * (2 * n) + W, "little")
    co = [int.from_bytes(raw[W * m:W * m + W], "little") for m in range(2 * n)]
    P = [(co[c] + co[c + n]) // 2 for c in range(n)]
    return min(k + P[(2 * k + n - 1) % n] for k in range(n))

for s, e in EXAMPLES:
    print(s, min_ops_bigint(s), e)

abc 2 2
yb 3 3


**Độ phức tạp.** `26` phép nhân số nguyên cỡ `24·n` bit; Karatsuba `O((24n/64)^1.585)` mỗi phép — thực tế vài trăm ms mỗi phép ở `n = 5·10^4`. Bộ nhớ `O(n)` số nguyên lớn.

## 8. TRACE từng bước

### 8.1. Trace theo `k` cho `s = "abcd"` (đáp án 3)

Cho mỗi `k`: chuỗi xoay, các cặp (đầu, cuối), chi phí từng cặp, tổng `k + chi phí`. Đồng thời in `c = (2k + n − 1) mod n` và `P[c]` tính theo cách gom tổng chỉ số để thấy hai cách khớp.

In [14]:
def trace_rotated_palindrome(s, A=26):
    n = len(s)
    a = to_nums(s)
    P = [0] * n
    pairs_by_sum = {c: [] for c in range(n)}
    for i in range(n):
        for j in range(i + 1, n):
            c = (i + j) % n
            P[c] += cyc_cost(a[i], a[j], A)
            pairs_by_sum[c].append((i, j))
    print("P[c] (gom theo tổng chỉ số):", P)
    for c in range(n):
        print(f"   c = {c}: các cặp chỉ số gốc {pairs_by_sum[c]}")
    best = float("inf")
    for k in range(n):
        rot = s[k:] + s[:k]
        c = (2 * k + n - 1) % n
        costs = [(rot[i], rot[n - 1 - i], cyc_cost(a[(i + k) % n], a[(n - 1 - i + k) % n], A)) for i in range(n // 2)]
        total = k + sum(x[2] for x in costs)
        assert total == k + P[c]
        best = min(best, total)
        print(f"k={k}: chuỗi {rot!r}, c={c}, cặp (đầu, cuối, chi phí) = {costs}, tổng = {k} + {sum(x[2] for x in costs)} = {total}")
    print("Đáp án =", best)
    return best

trace_rotated_palindrome("abcd")

P[c] (gom theo tổng chỉ số): [2, 2, 2, 4]
   c = 0: các cặp chỉ số gốc [(1, 3)]
   c = 1: các cặp chỉ số gốc [(0, 1), (2, 3)]
   c = 2: các cặp chỉ số gốc [(0, 2)]
   c = 3: các cặp chỉ số gốc [(0, 3), (1, 2)]
k=0: chuỗi 'abcd', c=3, cặp (đầu, cuối, chi phí) = [('a', 'd', 3), ('b', 'c', 1)], tổng = 0 + 4 = 4
k=1: chuỗi 'bcda', c=1, cặp (đầu, cuối, chi phí) = [('b', 'a', 1), ('c', 'd', 1)], tổng = 1 + 2 = 3
k=2: chuỗi 'cdab', c=3, cặp (đầu, cuối, chi phí) = [('c', 'b', 1), ('d', 'a', 3)], tổng = 2 + 4 = 6
k=3: chuỗi 'dabc', c=1, cặp (đầu, cuối, chi phí) = [('d', 'c', 1), ('a', 'b', 1)], tổng = 3 + 2 = 5
Đáp án = 3


3

### 8.2. Trace ví dụ 1 và ví dụ 2

Quan sát với `"yb"` (`n = 2`): chỉ có cặp `{0, 1}` với `c = 1` và chi phí `3`; `k = 0` cho `0 + 3`, `k = 1` cho `1 + 3 = 4` (vì với `n = 2`, mọi `k` đều dùng cùng một cặp).

In [15]:
trace_rotated_palindrome("abc")
trace_rotated_palindrome("yb")

P[c] (gom theo tổng chỉ số): [1, 1, 2]
   c = 0: các cặp chỉ số gốc [(1, 2)]
   c = 1: các cặp chỉ số gốc [(0, 1)]
   c = 2: các cặp chỉ số gốc [(0, 2)]
k=0: chuỗi 'abc', c=2, cặp (đầu, cuối, chi phí) = [('a', 'c', 2)], tổng = 0 + 2 = 2
k=1: chuỗi 'bca', c=1, cặp (đầu, cuối, chi phí) = [('b', 'a', 1)], tổng = 1 + 1 = 2
k=2: chuỗi 'cab', c=0, cặp (đầu, cuối, chi phí) = [('c', 'b', 1)], tổng = 2 + 1 = 3
Đáp án = 2
P[c] (gom theo tổng chỉ số): [0, 3]
   c = 0: các cặp chỉ số gốc []
   c = 1: các cặp chỉ số gốc [(0, 1)]
k=0: chuỗi 'yb', c=1, cặp (đầu, cuối, chi phí) = [('y', 'b', 3)], tổng = 0 + 3 = 3
k=1: chuỗi 'by', c=1, cặp (đầu, cuối, chi phí) = [('b', 'y', 3)], tổng = 1 + 3 = 4
Đáp án = 3


3

### 8.3. Trace "chéo hoá hạt nhân": kiểm `cost(d)` có đúng bằng tổng các `cos` không?

Với `A = 26`: `cost(d) = (1/26) Σ_{t=0}^{25} ĉ(t) cos(2π t d / 26)`. Gộp `t` và `26 − t` ta chỉ cần `t = 0..13` với hệ số `mult`.

In [16]:
A = 26
chat = [sum(min(z, A - z) * math.cos(2 * math.pi * t * z / A) for z in range(A)) for t in range(A + 1)]
print("ĉ(t), t=0..13:", [round(v, 3) for v in chat[:14]])
for d in range(0, 14):
    approx = sum((1 if t in (0, 13) else 2) * chat[t] * math.cos(2 * math.pi * t * d / A) for t in range(14)) / A
    assert abs(approx - min(d, A - d)) < 1e-9, d
print("cost(d) = (1/26) Σ_t mult_t · ĉ(t) · cos(2π t d / 26) đúng với d = 0..13")

ĉ(t), t=0..13: [169.0, -68.827, 0.0, -7.953, 0.0, -3.099, 0.0, -1.785, -0.0, -1.275, 0.0, -1.061, 0.0, -1.0]
cost(d) = (1/26) Σ_t mult_t · ĉ(t) · cos(2π t d / 26) đúng với d = 0..13


## 9. Code nộp LeetCode

Bản nộp dùng Cách 4 (FFT 14 tần số). LeetCode có sẵn `numpy`. Nếu muốn tránh `numpy`, có thể dùng Cách 5 (số nguyên lớn) ở mục 7.4.

In [17]:
import math
import numpy as np


class Solution:
    def minOperations(self, s: str) -> int:
        n = len(s)
        size = 1
        while size < 2 * n:
            size <<= 1

        nums = np.array([ord(c) - ord("a") for c in s], dtype=np.int64)

        # chat[t] = sum_z min(z, 26 - z) * cos(2*pi*t*z/26)  (DFT của hàm chi phí)
        chat = np.zeros(26)
        for t in range(26):
            for z in range(26):
                chat[t] += min(z, 26 - z) * math.cos(2 * math.pi * t * z / 26)

        dp = np.zeros(n)     # dp[c] ~ 52 * (tổng chi phí ghép cặp có tổng chỉ số ≡ c mod n)
        for t in range(14):
            theta = 2 * math.pi * t / 26
            a = np.exp(1j * theta * nums)
            fa = np.fft.fft(a, size)
            fb = np.fft.fft(np.conj(a), size)
            conv = np.fft.ifft(fa * fb).real
            mult = 1 if t == 0 or t == 13 else 2
            dp += mult * chat[t] * (conv[:n] + conv[n:2 * n])

        ans = float("inf")
        for k in range(n):
            c = (2 * k + n - 1) % n
            ans = min(ans, k + round(dp[c] / 52))
        return ans


sol = Solution()
for s, e in EXAMPLES:
    assert sol.minOperations(s) == e, s
print("2 ví dụ của đề: OK")

2 ví dụ của đề: OK


## 10. Vì sao thuật toán đúng

**(a) Mô hình `k + ghép cặp`.** Mọi dãy phép cho ra: chuỗi gốc xoay trái `k` lần (`k` = số phép xoay, và chỉ cần `0 ≤ k < n` vì `n` phép xoay quay về chuỗi cũ) với một số ký tự đã được tăng. Mỗi ký tự gốc chỉ tham gia đúng một cặp (hoặc là ký tự giữa), nên các chi phí tăng độc lập giữa các cặp. Với mỗi cặp `(x, y)`, chi phí tối thiểu để chúng bằng nhau chỉ bằng phép tăng là `min((y − x) mod 26, (x − y) mod 26) = min(d, 26 − d)`. (Mọi cách thứ tự phép đều có tổng số phép như vậy; BFS ở phần 12 xác nhận.)

**(b) Cặp theo tổng chỉ số.** Phần 6.1: cặp đối xứng sau `k` lần xoay là tập các cặp chỉ số gốc phân biệt có tổng `≡ c`. Điều này cho phép đưa `k` vào chỉ qua `c`.

**(c) Tích chập đếm đúng cặp.** `conv[m] = Σ_{i+j=m}` đếm cặp có thứ tự; gấp `m` và `m + n` cho tổng mod `n`. Cặp `i = j` chỉ xuất hiện khi `2i ≡ c`, có chi phí `cost(s[i], s[i]) = 0` nên không ảnh hưởng.

**(d) DFT chéo hoá đúng.** Mục 8.3 kiểm `cost(d) = (1/26) Σ ĉ(t) cos(2π t d/26)` số học. Vì `e^{iθ(x−y)}` có phần thực `cos(θ (x − y))`, tổng `Σ_{i+j=m} Re(...) = Re(conv)`, mà `conv` đã là số thực (đối xứng) nên lấy `.real` là đủ.

**(e) Làm tròn an toàn.** `dp[c]/52` bằng số nguyên cộng sai số `~10^-9`; làm tròn đúng khi sai số `< 0.5`.

## 11. Phân tích Time/Space

Đặt `n = len(s)`, `A = 26`.

| Cách | Time | Space |
|---|---|---|
| 1a. BFS | `O(A^n · n)` | `O(A^n)` |
| 1b. Công thức theo `k` | `O(n²)` | `O(n)` |
| 2. Gom tổng chỉ số | `O(n²)` | `O(n)` |
| 3. FFT 26 chỉ báo | `O(A² · n log n)` | `O(A · n)` |
| 4. FFT 14 tần số | `O(A · n log n)` | `O(n)` |
| 5. Nhân số nguyên lớn | `Õ(A · n^1.58)` (Karatsuba) | `O(n)` |

Ta đo thời gian ở `n = 5·10^4` và cũng đo **sai số làm tròn** của Cách 4.

In [18]:
import time, random
rng = random.Random(4028)
big = "".join(rng.choice("abcdefghijklmnopqrstuvwxyz") for _ in range(50000))
t = time.time(); r1 = Solution().minOperations(big); t1 = time.time() - t
t = time.time(); r2 = min_ops_fft_matrix(big); t2 = time.time() - t
t = time.time(); r3 = min_ops_bigint(big); t3 = time.time() - t
print(f"FFT 14 tần số : {r1} ({t1:.2f}s)")
print(f"FFT 26 chỉ báo: {r2} ({t2:.2f}s)")
print(f"số nguyên lớn : {r3} ({t3:.2f}s)")
assert r1 == r2 == r3

# Sai số làm tròn của cách 4 trên n = 5e4: đo max |dp/52 - round(dp/52)|
n = len(big)
size = 1
while size < 2 * n:
    size <<= 1
nums = np.array([ord(c) - 97 for c in big], dtype=np.int64)
chat = np.array([sum(min(z, 26 - z) * math.cos(2 * math.pi * t_ * z / 26) for z in range(26)) for t_ in range(26)])
dp = np.zeros(n)
for t_ in range(14):
    th = 2 * math.pi * t_ / 26
    a = np.exp(1j * th * nums)
    conv = np.fft.ifft(np.fft.fft(a, size) * np.fft.fft(np.conj(a), size)).real
    mult = 1 if t_ in (0, 13) else 2
    dp += mult * chat[t_] * (conv[:n] + conv[n:2 * n])
err = np.max(np.abs(dp / 52 - np.rint(dp / 52)))
print("sai số làm tròn lớn nhất:", err)
assert err < 0.01

FFT 14 tần số : 160619 (0.16s)
FFT 26 chỉ báo: 160619 (0.17s)
số nguyên lớn : 160619 (2.50s)
sai số làm tròn lớn nhất: 1.4551915228366852e-10


## 12. Bộ test

Ta kiểm: ví dụ đề, edge case, rồi đối chiếu. **BFS trên chuỗi thật** làm trọng tài: với bảng chữ cái thu nhỏ (`A = 3..6`, `n ≤ 6`) và với `A = 26` cho `n ≤ 3`. Các cách còn lại đều có tham số `A` nên so được với BFS ở cả hai cấu hình.

In [19]:
def rand_string(rng, n, A):
    return "".join(chr(97 + rng.randrange(A)) for _ in range(n))

# Edge cases (A = 26)
EDGE = [("aa", 0), ("ab", 1), ("az", 1), ("aaa", 0), ("aba", 0), ("abca", 1), ("zzzz", 0), ("ac", 2), ("an", 13)]
FUNCS26 = [min_ops_formula, min_ops_group, min_ops_fft_matrix, min_ops_fft_dft, min_ops_bigint, Solution().minOperations]
for s, e in EXAMPLES + EDGE:
    for fn in FUNCS26:
        assert fn(s) == e, (fn.__name__, s, fn(s), e)
    if len(s) <= 3:
        assert min_ops_bfs(s) == e, (s, min_ops_bfs(s), e)
print("Ví dụ đề + edge case: mọi cách đúng")

# BFS trọng tài, bảng chữ cái thu nhỏ
rng = random.Random(402801)
FUNCS_A = [min_ops_formula, min_ops_group, min_ops_fft_matrix, min_ops_fft_dft, min_ops_bigint]
for _ in range(400):
    A = rng.choice([3, 4, 5, 6])
    n = rng.randint(2, 6)
    s = rand_string(rng, n, A)
    ref = min_ops_bfs(s, A)
    for fn in FUNCS_A:
        assert fn(s, A) == ref, (fn.__name__, s, A, fn(s, A), ref)
print("400 chuỗi (A=3..6, n<=6): mọi cách khớp BFS")

for _ in range(60):
    n = rng.randint(2, 3)
    s = rand_string(rng, n, 26)
    ref = min_ops_bfs(s)
    assert Solution().minOperations(s) == ref == min_ops_formula(s), (s, ref)
print("60 chuỗi A=26, n<=3: Solution khớp BFS")

Ví dụ đề + edge case: mọi cách đúng


400 chuỗi (A=3..6, n<=6): mọi cách khớp BFS
60 chuỗi A=26, n<=3: Solution khớp BFS


In [20]:
# n lớn hơn, A = 26: đối chiếu công thức O(n^2) với các cách nhanh
rng = random.Random(7)
for _ in range(40):
    n = rng.randint(2, 60)
    s = rand_string(rng, n, 26)
    ref = min_ops_formula(s)
    assert min_ops_group(s) == ref
    assert min_ops_fft_matrix(s) == ref
    assert Solution().minOperations(s) == ref
    assert min_ops_bigint(s) == ref
print("40 chuỗi n<=60, A=26: các cách khớp công thức")

for n in (500, 1000):
    s = rand_string(rng, n, 26)
    ref = min_ops_group(s)
    assert Solution().minOperations(s) == ref == min_ops_bigint(s)
print("n = 500, 1000: Solution khớp gom tổng chỉ số O(n^2)")

40 chuỗi n<=60, A=26: các cách khớp công thức


n = 500, 1000: Solution khớp gom tổng chỉ số O(n^2)


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Chỉ tính chi phí `|x − y|`, quên đường vòng quanh `z → a`

**Code sai:** `abs(x - y)`.
**Vì sao sai:** `"yb"`: `|24 − 1| = 23` nhưng chỉ cần 3 phép (`y → z → a → b`).
**Code đúng:** `min(d, 26 - d)`.

In [21]:
def wrong_no_wrap(s):
    n = len(s); a = to_nums(s); best = float("inf")
    for k in range(n):
        best = min(best, k + sum(abs(a[(i + k) % n] - a[(n - 1 - i + k) % n]) for i in range(n // 2)))
    return best

print("sai:", wrong_no_wrap("yb"), "| đúng:", Solution().minOperations("yb"))

sai: 23 | đúng: 3


### Lỗi 2 — Quên cộng chi phí xoay `k`

**Code sai:** chỉ lấy `min_k P[...]`.
**Vì sao sai:** xoay không miễn phí. `"abc"`: bỏ `k` ra thì `k = 1` cho chi phí 1, nhưng đáp án thật là 2.
**Code đúng:** `k + P[(2k + n − 1) % n]`.

In [22]:
def wrong_free_rotation(s):
    n = len(s); a = to_nums(s)
    return min(sum(cyc_cost(a[(i + k) % n], a[(n - 1 - i + k) % n]) for i in range(n // 2)) for k in range(n))

print("sai:", wrong_free_rotation("abc"), "| đúng:", Solution().minOperations("abc"))

sai: 1 | đúng: 2


### Lỗi 3 — Ghép cặp sai chỉ số sau khi xoay (dùng `i` thay cho `(i + k) % n`)

**Code sai:** `cost(s[i], s[n - 1 - i])` cho mọi `k`.
**Vì sao sai:** sau xoay, ký tự ở vị trí `i` là `s[(i + k) % n]`.

### Lỗi 4 — Quên "gấp" tổng chỉ số `i + j ≥ n` về `(i + j) mod n`

**Code sai:** chỉ cộng chi phí vào `P[i + j]` khi `i + j < n` (hoặc chỉ lấy `conv[:n]` sau FFT mà không cộng `conv[n:2n]`).
**Vì sao sai:** các cặp có `i + j ≥ n` cũng là cặp đối xứng của một số phép xoay (tổng chỉ số được tính **theo mod `n`**), bỏ chúng làm `P` thiếu.
**Code đúng:** `P[(i + j) % n]`; với FFT, `conv[:n] + conv[n:2n]`.

In [23]:
def wrong_no_fold(s):
    n = len(s); a = to_nums(s)
    P = [0] * n
    for i in range(n):
        for j in range(i + 1, n):
            if i + j < n:                          # SAI: bỏ các cặp có i + j >= n
                P[i + j] += cyc_cost(a[i], a[j])
    return min(k + P[(2 * k + n - 1) % n] for k in range(n))

rng = random.Random(8)
for _ in range(2000):
    s = rand_string(rng, rng.randint(2, 6), 26)
    if wrong_no_fold(s) != Solution().minOperations(s):
        print("phản ví dụ:", s, "| sai:", wrong_no_fold(s), "| đúng:", Solution().minOperations(s))
        break

phản ví dụ: wbc | sai: 2 | đúng: 3


### Lỗi 5 — Chia sai hằng số khi lấy lại `P`

**Code sai:** `round(dp[c] / 26)`.
**Vì sao sai:** có hai nhân tử: DFT chia `26`, và cặp có thứ tự đếm đôi nên chia thêm `2`. Chia `26` cho kết quả gấp đôi.
**Code đúng:** `round(dp[c] / 52)` (tổng quát `2A`).

## 14. Mẹo & nhận dạng pattern

- **"Với mỗi phép dịch/xoay, tính tổng độ lệch của các cặp"** → tìm một đại lượng bất biến theo cặp (ở đây: **tổng chỉ số mod n**), rồi dồn thành **tích chập**.
- **Chi phí chỉ phụ thuộc hiệu trên một nhóm tuần hoàn** (chữ cái, góc, giờ) → khai triển Fourier hạt nhân chi phí, số tần số cần dùng nhỏ.
- **FFT bằng `numpy`**: đệm số 0, làm tròn bằng `np.rint`, kiểm sai số thật sự (như mục 11).
- **Không có numpy?** Nhân số nguyên lớn của Python (gói các hệ số vào các khối bit) cho tích chập chính xác với mọi hệ số nhỏ.
- Với bài tối ưu, luôn có trọng tài **BFS trên bảng chữ cái thu nhỏ**: mọi công thức có tham số `A` đều kiểm thử được.
- `n` lẻ: ký tự giữa miễn phí, cặp `i = j` có chi phí 0 nên không cần xử lý riêng.

## 15. Biến thể

1. **Cho phép xoay phải** (cost `min(k, n − k)` cho xoay): sửa chi phí xoay, giữ nguyên `P` (Bài 5, đã đối chiếu BFS).
2. **Cả tăng và giảm ký tự:** chi phí `min(d, 26 − d)` vẫn đúng, vì chọn đường ngắn nhất cả hai chiều.
3. **Chỉ tăng, không xoay** (Rotated Palindrome không có xoay): `P` chỉ tại `k = 0`, `O(n)`.
4. **Bảng chữ cái `A` khác 26:** thay `26` bằng `A`, `52` bằng `2A` — `min_ops_fft_dft(s, A)` đã tham số hoá.
5. **Chi phí tuỳ ý cho cặp chữ cái** (không chỉ cung ngắn): dùng Cách 3 với ma trận chi phí bất kỳ (26 FFT).

## 16. Bảng so sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1a. BFS | Trạng thái là chuỗi | `O(A^n n)` | `O(A^n)` | Trọng tài, `A` nhỏ |
| 1b. Công thức theo `k` | Ghép lại mỗi `k` | `O(n²)` | `O(n)` | Kiểm chứng, `n ≤ 2000` |
| 2. Gom tổng chỉ số | `P[c]` qua mọi cặp | `O(n²)` | `O(n)` | Cầu nối sang tích chập |
| 3. FFT 26 chỉ báo | `Σ cost(x,y) conv(I_x, I_y)` | `O(A² n log n)` | `O(A n)` | Chi phí bất kỳ |
| 4. FFT 14 tần số | DFT chéo hoá hạt nhân cung ngắn | `O(A n log n)` | `O(n)` | **Bản nộp** |
| 5. Số nguyên lớn | Kronecker, chính xác | `Õ(A n^1.58)` | `O(n)` | Không có numpy |

## 17. Bài tập (5 bài, từ dễ đến khó)

**Bài 1 (đoán output).** `cyc_cost(24, 1)` bằng bao nhiêu? Giải thích bằng hình "kim đồng hồ".

**Bài 2 (tính tay).** `s = "abcd"`. Lập bảng `k, chuỗi xoay, chi phí cặp, tổng` và cho đáp án.

**Bài 3 (điền TODO).** Hoàn thiện `pair_sums(s)` trả về dict `{c: [danh sách cặp (i, j) với i < j, (i + j) % n == c]}`.

**Bài 4 (tự viết lại).** Viết `my_min_ops(s)` theo cách gom tổng chỉ số (Cách 2), tức là `min_k (k + P[(2k + n − 1) % n])`.

**Bài 5 (biến thể).** Nếu cho phép **cả xoay phải** (mỗi lần xoay phải tốn 1 phép), chi phí xoay `k` vị trí là `min(k, n − k)`. Viết `min_ops_both_rotations(s)` và đối chiếu BFS có thêm phép xoay phải.

In [24]:
# Bài 3 — TODO (không gọi khi chạy notebook)
def pair_sums(s):
    # TODO 1: n = len(s); tạo dict {c: [] for c in range(n)}
    # TODO 2: duyệt i < j; c = (i + j) % n; thêm (i, j) vào dict[c]
    pass

# Bài 4 — TODO
def my_min_ops(s):
    # TODO: P[c] = tổng cyc_cost của các cặp trong pair_sums(s)[c]; trả về min_k (k + P[(2*k+n-1) % n])
    pass

# Bài 5 — TODO
def min_ops_both_rotations(s):
    # TODO: như bài 4 nhưng thay k bằng min(k, n - k)
    pass

## 18. Gợi ý 3 mức

**Bài 1.**
- Mức 1: `d = |24 − 1| = 23`.
- Mức 2: đi tiếp theo chiều tăng từ `y` (24) đến `b` (1 + 26 = 27) mất `27 − 24 = 3`.
- Mức 3: `min(23, 26 − 23) = 3`.

**Bài 2.**
- Mức 1: `n = 4`, hai cặp mỗi lần.
- Mức 2: `k = 0`: `abcd` cặp (a,d)=3, (b,c)=1 → 4. `k = 1`: `bcda` cặp (b,a)=1, (c,d)=1 → `1 + 2 = 3`.
- Mức 3: `k = 2`: `cdab` → (c,b)=1, (d,a)=3 → `2 + 4 = 6`; `k = 3`: `dabc` → (d,c)=1,(a,b)=1 → `3 + 2 = 5`. Đáp án **3**.

**Bài 3.**
- Mức 1: hai vòng `for` lồng nhau.
- Mức 2: `for j in range(i + 1, n)`.
- Mức 3: `d[(i + j) % n].append((i, j))`.

**Bài 4.**
- Mức 1: dùng kết quả bài 3 hoặc cộng thẳng vào `P`.
- Mức 2: `c = (2*k + n - 1) % n`.
- Mức 3: `min(k + P[c] for k in range(n))`.

**Bài 5.**
- Mức 1: chỉ đổi phần chi phí xoay.
- Mức 2: xoay phải `n − k` lần cho kết quả giống xoay trái `k` lần.
- Mức 3: `min(min(k, n - k) + P[c] ...)`.

## 19. Đáp án tham khảo

In [25]:
# Bài 1, 2
assert cyc_cost(24, 1) == 3
assert Solution().minOperations("abcd") == 3

# Bài 3
def pair_sums_ans(s):
    n = len(s)
    d = {c: [] for c in range(n)}
    for i in range(n):
        for j in range(i + 1, n):
            d[(i + j) % n].append((i, j))
    return d

ps = pair_sums_ans("abcd")
assert sum(len(v) for v in ps.values()) == 6          # C(4,2)
assert all((i + j) % 4 == c for c, v in ps.items() for i, j in v)
print("Bài 1–3 OK")

Bài 1–3 OK


In [26]:
# Bài 4
def my_min_ops_ans(s):
    n = len(s); a = to_nums(s)
    ps = pair_sums_ans(s)
    P = [sum(cyc_cost(a[i], a[j]) for i, j in ps[c]) for c in range(n)]
    return min(k + P[(2 * k + n - 1) % n] for k in range(n))

# Bài 5
def min_ops_both_rotations_ans(s, A=26):
    n = len(s); a = to_nums(s)
    P = [0] * n
    for i in range(n):
        for j in range(i + 1, n):
            P[(i + j) % n] += cyc_cost(a[i], a[j], A)
    # xoay phải n - k lần cho cùng kết quả như xoay trái k lần
    return min(min(k, n - k) + P[(2 * k + n - 1) % n] for k in range(n))

rng = random.Random(99)
for _ in range(300):
    n = rng.randint(2, 6)
    A = rng.choice([3, 4, 5])
    s = rand_string(rng, n, A)
    assert min_ops_both_rotations_ans(s, A) == min_ops_bfs(s, A, right_rotate=True), (s, A)
for _ in range(100):
    s = rand_string(rng, rng.randint(2, 12), 26)
    assert my_min_ops_ans(s) == Solution().minOperations(s)
print("Bài 4–5 OK (bài 5 đối chiếu BFS có xoay phải)")

Bài 4–5 OK (bài 5 đối chiếu BFS có xoay phải)


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template:**

```python
cost(x, y) = min(|x - y|, 26 - |x - y|)
c(k)       = (2k + n - 1) % n                    # tổng chỉ số của mọi cặp sau k lần xoay
P[c]       = sum cost(s[i], s[j]) với i < j, (i + j) % n == c
answer     = min over k in [0, n) of  k + P[c(k)]

# tính P cho mọi c nhanh:
conv_t = ifft(fft(a_t) * fft(conj(a_t)))   với a_t[i] = exp(i * 2*pi*t/26 * s[i]),  t = 0..13
dp[c]  = sum_t mult_t * chat[t] * (conv_t[c] + conv_t[c + n]);   P[c] = round(dp[c] / 52)
```

**Checklist trước khi submit:**

- [ ] Chi phí cặp là `min(d, 26 - d)` (đường vòng quanh `z → a`).
- [ ] Đã cộng chi phí xoay `k` và chỉ xét `k` trong `[0, n)`.
- [ ] `c = (2k + n - 1) % n` và gấp `conv[m] + conv[m + n]`.
- [ ] Đệm số 0 tới `size ≥ 2n` trước FFT.
- [ ] Hệ số `mult` (1 cho `t = 0, 13`, 2 còn lại) và chia `52`.
- [ ] `round` kết quả; đã kiểm sai số.
- [ ] Đã đối chiếu BFS trên bảng chữ cái nhỏ.